In [ ]:
import numpy as np
import pandas as pd 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# I. Tổng quan bộ dữ liệu
Nghiên cứu sử dụng dữ liệu thương mại quốc tế được thu thập từ cơ sở dữ liệu UN Comtrade do Liên Hợp Quốc công bố. Bộ dữ liệu phản ánh hoạt động xuất khẩu của Việt Nam trong giai đoạn 2019–2023, được phân loại theo Hệ thống Mã số Hài hòa (HS – Harmonized System) ở cấp độ 2 chữ số, tức nhóm ngành hàng.
Dữ liệu ban đầu bao gồm thông tin về giá trị xuất khẩu, quốc gia báo cáo, đối tác thương mại, mã hàng hóa và thời gian giao dịch. Đây là các thông tin có ý nghĩa quan trọng trong việc đánh giá quy mô và cơ cấu hoạt động xuất khẩu – những yếu tố mà ngân hàng cần xem xét khi thực hiện nghiệp vụ thông báo và xác nhận thư tín dụng (Letter of Credit – LC).

Sau quá trình lọc và xử lý, bộ dữ liệu được cấu trúc lại nhằm phục vụ cho phân tích mô tả và phân tích khám phá theo hướng Business Analytics, hướng tới việc trích xuất các chỉ báo hỗ trợ đánh giá quy mô giao dịch và mức độ rủi ro theo nhóm ngành xuất khẩu.

# II. Data Understanding

Trước khi tiến hành phân tích, cần hiểu rõ cấu trúc và đặc điểm của bộ dữ liệu UN Comtrade.  
Phần này nhằm:
- Xác định các biến chính trong bộ dữ liệu
- Hiểu ý nghĩa nghiệp vụ của từng biến
- Đánh giá sơ bộ phạm vi và mức độ đầy đủ của dữ liệu

## 2.1 Cấu trúc tổng thể bộ dữ liệu

Bộ dữ liệu sau khi tải về bao gồm các trường thông tin chính như sau:

In [ ]:
df = pd.read_excel("/kaggle/input/datasets/huonggianguet/tradedata/TradeData.xlsx") 

In [ ]:
df.head()

In [ ]:
df.info()

Bộ dữ liệu sau khi thu thập gồm 30.217 bản ghi với 47 biến, phản ánh hoạt động thương mại quốc tế của Việt Nam giai đoạn 2019–2023. Cấu trúc dữ liệu mang tính đa chiều, bao gồm biến định danh, biến phân loại và biến định lượng, cho phép phân tích theo nhiều góc độ như thời gian, ngành hàng và giá trị giao dịch.

Các biến chủ yếu thuộc ba dạng: chuỗi (mô tả quốc gia, luồng thương mại, hàng hóa), số nguyên/số thực (thời gian, trọng lượng, giá trị) và logic (đánh dấu trạng thái ước tính). Một số biến chi tiết như đơn vị số lượng hoặc giá trị CIF chưa đầy đủ, trong khi biến primaryValue có dữ liệu hoàn chỉnh và được chọn làm đại diện cho giá trị xuất khẩu.

Tổng thể, bộ dữ liệu UN Comtrade đảm bảo nền tảng đủ tin cậy để phân tích quy mô, cơ cấu và xu hướng xuất khẩu, dù vẫn tồn tại hạn chế ở một số biến chi tiết. Điều này tạo cơ sở cho các bước tiền xử lý và phân tích tiếp theo nhằm phục vụ đánh giá quy mô giao dịch và rủi ro theo ngành hàng trong nghiệp vụ thư tín dụng.

## 2.2 Phạm vi quốc gia và thời gian nghiên cứu

Nghiên cứu tập trung vào hoạt động xuất khẩu của Việt Nam trong giai đoạn 2019–2023.  
Khoảng thời gian này bao gồm cả giai đoạn trước, trong và sau đại dịch COVID-19, giúp quan sát sự biến động của hoạt động thương mại quốc tế.


In [ ]:
df['refYear'].value_counts().sort_index()
df['reporterISO'].value_counts().head()

# III. Data Cleaning & Preparation

Dựa trên mục tiêu nghiên cứu và phạm vi đã xác định, bộ dữ liệu được lọc và xử lý theo các tiêu chí:
- Quốc gia báo cáo: Việt Nam
- Luồng thương mại: Xuất khẩu
- Giai đoạn: 2019–2023
- Phân loại hàng hóa: HS 2 chữ số
- Đối tác thương mại: Tổng thế giới

In [ ]:
print("Raw shape:", df.shape)
df.head()

## 3.1. Xác định và kiểm tra phạm vi dữ liệu nghiên cứu 

In [ ]:
# Lọc lại df_scope với partnerISO = W00
mask = (
    df["reporterISO"].isin(["VNM", "VN"]) &
    df["flowDesc"].str.contains("Export", case=False, na=False) &
    df["refYear"].between(2019, 2023) &
    (df["partnerISO"] == "W00")  # chỉ lấy tổng thế giới
)

df_scope = df.loc[mask].copy()  

print("After scope filtering:", df_scope.shape)
df_scope[["refYear", "reporterISO", "flowDesc","partnerISO"]].drop_duplicates().head(10)

Dữ liệu được lọc theo phạm vi nghiên cứu gồm: Việt Nam (reporterISO = VNM), luồng xuất khẩu (Export), đối tác thương mại: Tổng thế giới (partnerISO = W00) và giai đoạn 2019–2023.  
Kết quả kiểm tra cho thấy dữ liệu sau lọc chỉ chứa các năm trong phạm vi nghiên cứu và đúng luồng giao dịch xuất khẩu, đảm bảo tính nhất quán trước khi xử lý sâu hơn.


## 3.2. Kiểm tra độ đầy đủ của các cột giá trị 

In [ ]:
value_cols = ["primaryValue", "fobvalue", "cifvalue"]
missing_report = df_scope[value_cols].isna().sum().to_frame("missing_count")
missing_report["missing_rate"] = missing_report["missing_count"] / len(df_scope)
missing_report.sort_values("missing_rate", ascending=False)

Kết quả kiểm tra mức độ thiếu dữ liệu cho thấy sự khác biệt rõ rệt giữa các biến giá trị giao dịch.  
Trong khi biến `primaryValue` và `fobvalue` có dữ liệu đầy đủ cho toàn bộ các bản ghi, biến `cifvalue` lại thiếu dữ liệu ở hơn 60% số giao dịch.

Do tỷ lệ thiếu dữ liệu của `cifvalue` ở mức cao, việc sử dụng biến này trong phân tích tổng thể có thể dẫn đến sai lệch và làm giảm tính đại diện của kết quả.  
Vì vậy, trong phạm vi nghiên cứu này, `primaryValue` được lựa chọn làm biến đại diện cho giá trị xuất khẩu nhằm đảm bảo tính nhất quán và độ tin cậy của các phân tích tiếp theo.


## 3.3. Lựa chọn các biến phục vụ phân tích

In [ ]:
keep_cols = [
    "refYear",       # Năm
    "cmdCode",       # Mã hàng (HS)
    "cmdDesc",       # Mô tả hàng hóa
    "aggrLevel",     # Mức độ tổng hợp 
    "primaryValue",  # Giá trị giao dịch 
]
keep_cols = [c for c in keep_cols if c in df_scope.columns]  # tránh lỗi nếu thiếu cột

df_prepared = df_scope[keep_cols].copy()
print("Selected shape:", df_prepared.shape)
df_prepared.head()

Sau khi xác định phạm vi nghiên cứu và kiểm tra mức độ đầy đủ của các biến giá trị, bộ dữ liệu được tiếp tục tinh giản bằng cách lựa chọn các biến phục vụ trực tiếp cho mục tiêu phân tích. Cụ thể, các biến được giữ lại bao gồm thông tin về thời gian giao dịch, mã và mô tả hàng hóa, mức độ tổng hợp theo HS, cùng với giá trị giao dịch xuất khẩu. Việc lựa chọn này giúp giảm độ phức tạp của dữ liệu, đồng thời vẫn đảm bảo đầy đủ thông tin cần thiết để phân tích xu hướng, cơ cấu ngành hàng và mức độ tập trung xuất khẩu.

In [ ]:
df_prepared.rename(columns={
    "refYear": "Year",
    "cmdCode": "HS_Code",
    "cmdDesc": "HS_Desc",
    "primaryValue": "Trade_Value"
}, inplace=True)

df_prepared.info()

Trong phạm vi nghiên cứu, dữ liệu được tinh giản chỉ giữ lại các trường cần thiết để phân tích theo năm và theo nhóm ngành hàng: năm (Year), mã HS (HS_Code), mô tả ngành (HS_Desc) và giá trị giao dịch (Trade_Value).  
Việc đổi tên cột giúp tăng tính nhất quán và dễ đọc trong quá trình phân tích và trực quan hóa, đặc biệt khi triển khai trên Power BI.


## 3.4. Chuẩn hóa HS về HS 2-digit

Dữ liệu gốc đã ở cấp HS 2 chữ số, bước xử lý này chỉ nhằm chuẩn hóa cách biểu diễn mã ngành về dạng 2 ký tự, ví dụ 1 thành 01.

In [ ]:
def format_hs2_code(x):
    if pd.isna(x):
        return np.nan

    s = str(int(x))
    return s.zfill(2)


df_prepared["HS2"] = df_prepared["HS_Code"].apply(format_hs2_code)

hs2_summary = (
    df_prepared["HS2"]
    .value_counts()
    .rename("Số bản ghi")
    .to_frame()
)

hs2_summary["Tỷ trọng (%)"] = (
    hs2_summary["Số bản ghi"] / hs2_summary["Số bản ghi"].sum() * 100
)

hs2_summary = hs2_summary.reset_index().rename(columns={"index": "HS 2-digit"})

hs2_summary.head(10).style.format({
    "Tỷ trọng (%)": "{:.2f}%"
})

Kết quả trên cho thấy dữ liệu xuất khẩu của Việt Nam được phân bố trên nhiều nhóm ngành hàng khác nhau theo phân loại HS 2 chữ số.  

## 3.5. Chuẩn hóa mô tả ngành HS
Trong dữ liệu gốc, một số mã HS có nhiều mô tả khác nhau do khác biệt về cách diễn đạt như bạn có thể thấy dưới đây 

In [ ]:
desc_check = (
    df_prepared.groupby("HS2")["HS_Desc"]
    .nunique()
    .reset_index()
)

desc_check[desc_check["HS_Desc"] > 1]

In [ ]:
hs_desc_multi = (
    df_prepared.groupby("HS2")["HS_Desc"]
    .agg(lambda x: sorted(set(x.dropna())))
    .reset_index()
)

hs_desc_multi["Desc_Count"] = hs_desc_multi["HS_Desc"].apply(len)
hs_desc_multi = hs_desc_multi[hs_desc_multi["Desc_Count"] > 1].copy()

hs_desc_multi = hs_desc_multi.explode("HS_Desc").sort_values(["HS2", "HS_Desc"]).reset_index(drop=True)

hs_desc_multi

Để đảm bảo tính nhất quán khi hiển thị và phân tích, mỗi mã HS được gán một mô tả đại diện duy nhất.

In [ ]:
hs_desc_standard = (
    df_prepared
    .groupby("HS2")["HS_Desc"]
    .last()
    .reset_index()
)

df_prepared = df_prepared.drop(columns="HS_Desc").merge(
    hs_desc_standard,
    on="HS2",
    how="left"
)

In [ ]:
df_prepared.groupby("HS2")["HS_Desc"].nunique().value_counts()

In [ ]:
df_prepared[df_prepared["HS2"].isin(["15","16","24","84","88"])][["HS2","HS_Desc"]].drop_duplicates()

## 3.6. Xử lý giá trị & kiểu dữ liệu
 - Mục tiêu: đảm bảo Trade_Value là số và không âm.

In [ ]:
df_prepared["Trade_Value"] = pd.to_numeric(df_prepared["Trade_Value"], errors="coerce")

before = len(df_prepared)
df_prepared = df_prepared.dropna(subset=["Trade_Value"])
after = len(df_prepared)
print(f"Dropped rows with missing Trade_Value: {before - after}")

neg_count = (df_prepared["Trade_Value"] < 0).sum()
print("Negative Trade_Value count:", neg_count)

df_prepared = df_prepared[df_prepared["Trade_Value"] >= 0].copy()

df_prepared[["Year", "HS2", "Trade_Value"]].describe(include="all")

Sau khi chuẩn hóa kiểu dữ liệu, biến `Trade_Value` được chuyển về dạng số để đảm bảo có thể sử dụng trong các phép tổng hợp và trực quan hóa. Kết quả cho thấy dữ liệu không có bản ghi bị thiếu `Trade_Value` và không tồn tại giá trị âm. Điều này giúp đảm bảo độ tin cậy khi sử dụng `Trade_Value` làm biến đại diện cho quy mô xuất khẩu trong các phân tích tiếp theo.

Thống kê mô tả cũng cho thấy dữ liệu bao gồm nhiều nhóm ngành HS 2 chữ số (97 nhóm), trong đó có một số nhóm xuất hiện thường xuyên hơn. Đặc biệt, sự chênh lệch lớn giữa giá trị nhỏ nhất và lớn nhất phản ánh đặc điểm phân bố lệch của dữ liệu thương mại, do một số ngành có quy mô xuất khẩu vượt trội so với phần còn lại.


## 3.7. Tổng hợp dữ liệu theo năm và nhóm ngành hàng

In [ ]:
df_grouped = (
    df_prepared
    .groupby(["Year", "HS2"], as_index=False)
    .agg(
        Total_Export_Value=("Trade_Value", "sum"),
        Record_Count=("Trade_Value", "size")
    )
)

df_grouped = df_grouped.sort_values(["Year", "Total_Export_Value"], ascending=[True, False])

print("Aggregated shape:", df_grouped.shape)
df_grouped.head(10)


Sau khi tổng hợp dữ liệu theo năm và nhóm ngành hàng (HS 2 chữ số), bảng dữ liệu thu được gồm 484 dòng và 4 cột.  Mỗi dòng đại diện cho một nhóm ngành cụ thể trong một năm, kèm theo tổng giá trị xuất khẩu và số lượng bản ghi gốc được cộng lại để tạo nên giá trị này. Cấu trúc dữ liệu này giúp chuyển dữ liệu từ mức chi tiết sang mức tổng hợp, phù hợp cho việc phân tích xu hướng, cơ cấu ngành và trực quan hóa trong các bước tiếp theo.

## 3.8. Tạo biến “Industry_Group” để phân tích theo nhóm ngành lớn
Để tăng khả năng diễn giải theo góc nhìn nghiệp vụ, các mã HS2 được ánh xạ sang nhóm ngành lớn (Industry_Group).

In [ ]:
industry_map = {
    'Nông - Thủy sản':               ['01','02','03','04','05','06','07','08','09','10','11','12','13','14'],
    'Thực phẩm chế biến':            ['15','16','17','18','19','20','21','22','23','24'],
    'Khoáng sản & Năng lượng':       ['25','26','27'],
    'Hóa chất & Nhựa':               ['28','29','30','31','32','33','34','35','36','37','38','39','40'],
    'Dệt may - Da giày':             ['41','42','43','50','51','52','53','54','55','56','57','58','59','60','61','62','63','64','65','66','67'],
    'Gỗ - Giấy - Nội thất':          ['44','47','48','94'],
    'Đá - Thủy tinh - VL xây dựng':  ['68','69','70'],
    'Kim loại':                      ['71','72','73','74','75','76','78','79','80','81','82','83'],
    'Máy móc - Điện tử':             ['84','85'],
    'Phương tiện vận tải':           ['86','87','88','89'],
    'Dụng cụ & Thiết bị chuyên dụng':['90','91','92'],
    'Hàng tiêu dùng':                ['45','46','49','95','96'],
}

def assign_industry(hs2):
    hs2 = str(hs2).zfill(2)
    for group, codes in industry_map.items():
        if hs2 in codes:
            return group
    return "Khác"

df_grouped["Industry_Group"] = df_grouped["HS2"].apply(assign_industry)

# check
df_grouped["Industry_Group"].value_counts()

In [ ]:
# Kiểm tra tỷ trọng nhóm "Khác" — các mã HS không được phân loại
khac = df_grouped[df_grouped["Industry_Group"] == "Khác"]

tong_xk = df_grouped["Total_Export_Value"].sum()
tong_khac = khac["Total_Export_Value"].sum()
ty_trong_khac = tong_khac / tong_xk * 100

print(f"Số mã HS2 thuộc nhóm 'Khác': {khac['HS2'].nunique()}")
print(f"Các mã HS2 đó là: {sorted(khac['HS2'].unique().tolist())}")
print(f"Tổng giá trị xuất khẩu nhóm 'Khác': {tong_khac/1e6:,.1f} triệu USD")
print(f"Tỷ trọng trong tổng xuất khẩu: {ty_trong_khac:.3f}%")

Bước này giúp giảm độ chi tiết khi phân tích, phù hợp với mục tiêu đánh giá cơ cấu và mức độ ổn định theo ngành trong bối cảnh nghiệp vụ ngân hàng. Ngoài ra ta có thể thấy Nhóm 'Khác' chiếm dưới 2%, ảnh hưởng không đáng kể đến dữ liệu.

In [ ]:
hs_list = (
    df_prepared[["HS2", "HS_Desc"]]
    .drop_duplicates()
    .sort_values("HS2")
    .reset_index(drop=True)
)

hs_list.head(20)

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", None)        
pd.set_option("display.max_columns", None)     
pd.set_option("display.max_colwidth", None)    
pd.set_option("display.width", 1000)           

hs_list = (
    df_prepared[["HS2", "HS_Desc"]]
    .drop_duplicates()
    .sort_values("HS2")
    .reset_index(drop=True)
)

display(hs_list)

Danh mục HS2 – HS_Desc được tách riêng nhằm phục vụ tra cứu và hiển thị nhãn ngành trong dashboard Power BI. Toàn bộ danh mục có thể được đưa vào phần phụ lục để đảm bảo tính minh bạch và hỗ trợ người đọc hiểu rõ ý nghĩa các mã HS sử dụng trong nghiên cứu.

In [ ]:
#check
print("df_grouped:", df_grouped.shape)
print("hs_list:", hs_list.shape)

df_grouped.head()

# IV. Kết quả phân tích dữ liệu xuất khẩu Việt Nam (2019–2023)
## 4.1. Phân tích tổng quan quy mô xuất khẩu theo thời gian

Để đánh giá xu hướng phát triển của hoạt động xuất khẩu, tổng giá trị xuất khẩu theo năm được tính bằng cách cộng tổng giá trị xuất khẩu của tất cả các ngành HS2 trong từng năm. Tốc độ tăng trưởng xuất khẩu được đo lường thông qua chỉ số Year-over-Year Growth (YoY Growth).

**YoY_Growth_% = (Xuất khẩu năm N - Xuất khẩu năm N-1) / Xuất khẩu năm N-1 × 100**

In [ ]:
dfA = df_grouped.copy()

dfA["Year"] = dfA["Year"].astype(int)
dfA["Total_Export_Value"] = pd.to_numeric(dfA["Total_Export_Value"], errors="coerce")

dfA.head()

In [ ]:
# Tổng giá trị xuất khẩu theo năm
total_by_year = (
    dfA.groupby("Year", as_index=False)
       .agg(Total_Export=("Total_Export_Value", "sum"))
       .sort_values("Year")
)

total_by_year["YoY_Growth_%"] = total_by_year["Total_Export"].pct_change() * 100
total_by_year

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
ax.plot(total_by_year["Year"], total_by_year["Total_Export"] / 1e9,
        marker="o", color="#1d6fa4", linewidth=2)

for _, row in total_by_year.iterrows():
    ax.annotate(f'{row["Total_Export"]/1e9:.0f}B',
                xy=(row["Year"], row["Total_Export"]/1e9),
                xytext=(0, 10), textcoords="offset points",
                ha="center", fontsize=9)

ax.set_xlabel("Năm")
ax.set_ylabel("Tổng giá trị xuất khẩu (tỷ USD)")
ax.set_title("Tổng giá trị xuất khẩu của Việt Nam theo năm (2019–2023)")
ax.set_xticks(sorted(total_by_year["Year"].unique()))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("export_trend.png", dpi=150, bbox_inches="tight")
plt.show()

Trong giai đoạn 2019–2023, tổng giá trị xuất khẩu của Việt Nam ghi nhận xu hướng tăng liên tục từ năm 2019 đến 2022, trước khi điều chỉnh giảm vào năm 2023. Cụ thể, kim ngạch xuất khẩu tăng từ khoảng 265 tỷ USD năm 2019 lên mức đỉnh 371 tỷ USD vào năm 2022, tương đương mức tăng tích lũy khoảng 40% trong vòng ba năm. Sang năm 2023, kim ngạch giảm xuống còn khoảng 353 tỷ USD, tương ứng tốc độ tăng trưởng âm khoảng 4,8% so với năm trước.

Xét theo từng giai đoạn, năm 2020 dù chịu tác động trực tiếp của đại dịch COVID-19 và các đứt gãy trong chuỗi cung ứng toàn cầu, xuất khẩu Việt Nam vẫn duy trì mức tăng trưởng dương khoảng 6,4%, đạt khoảng 281 tỷ USD. Điều này phản ánh khả năng thích ứng tương đối tốt của nền kinh tế, đặc biệt là các ngành xuất khẩu chủ lực như điện tử và sản xuất công nghiệp, vốn ít bị gián đoạn hơn so với các lĩnh vực phụ thuộc vào dịch vụ hoặc tiêu dùng trực tiếp.

Giai đoạn 2021–2022 chứng kiến đà tăng trưởng mạnh mẽ với tốc độ tăng trưởng YoY lần lượt đạt khoảng 19,3% và 10,5%. Đây là giai đoạn phục hồi sau đại dịch khi nhu cầu hàng hóa toàn cầu tăng cao, đồng thời Việt Nam hưởng lợi từ xu hướng dịch chuyển chuỗi cung ứng và gia tăng vai trò trong mạng lưới sản xuất khu vực. Tuy nhiên, đến năm 2023, kim ngạch xuất khẩu ghi nhận sự suy giảm, chủ yếu do nhu cầu tiêu dùng toàn cầu yếu đi dưới tác động của lạm phát kéo dài tại các thị trường lớn như Mỹ và EU, đặc biệt ảnh hưởng đến các ngành xuất khẩu chủ lực như điện tử và dệt may.

Nhìn chung, xu hướng này cho thấy hoạt động xuất khẩu của Việt Nam có nền tảng tăng trưởng tương đối vững trong trung hạn, nhưng vẫn chịu ảnh hưởng lớn từ biến động của thị trường quốc tế. Đối với hoạt động tài trợ thương mại tại ngân hàng, giai đoạn tăng trưởng mạnh 2021–2022 phản ánh nhu cầu sử dụng thư tín dụng (L/C) gia tăng, trong khi sự điều chỉnh năm 2023 đặt ra yêu cầu các ngân hàng cần thận trọng hơn trong việc đánh giá rủi ro, đặc biệt liên quan đến khả năng thực hiện hợp đồng và năng lực tài chính của doanh nghiệp xuất khẩu.

## 4.2. Phân tích cơ cấu ngành hàng xuất khẩu
### 4.2.1 Cơ cấu theo nhóm ngành lớn (Industry_Group)

Để hiểu rõ hơn cấu trúc của hoạt động xuất khẩu, dữ liệu được tổng hợp theo các nhóm ngành lớn (Industry_Group) dựa trên việc phân loại các mã HS2 thành các nhóm ngành có đặc điểm kinh tế tương đồng. Việc phân nhóm này giúp giảm độ phân tán của dữ liệu HS2 và cho phép quan sát rõ hơn vai trò của từng nhóm ngành trong tổng kim ngạch xuất khẩu theo thời gian.

**Export_Share_% ngành X năm Y = Xuất khẩu(X, Y) / Tổng xuất khẩu(Y) × 100**

In [ ]:
group_by_year = (
    dfA.groupby(["Year", "Industry_Group"], as_index=False)
       .agg(Group_Export=("Total_Export_Value", "sum"))
)

# tính tỷ trọng trong năm
total_year = group_by_year.groupby("Year")["Group_Export"].transform("sum")
group_by_year["Export_Share_%"] = group_by_year["Group_Export"] / total_year * 100

group_by_year.sort_values(["Year", "Export_Share_%"], ascending=[True, False]).head(15)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

pivot_df = (
    group_by_year
    .pivot(index="Year", columns="Industry_Group", values="Export_Share_%")
    .fillna(0)
    .sort_index()
)

pivot_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)

pivot_df.plot(kind="bar", stacked=True, ax=ax,
              colormap="tab20", width=0.6)

ax.set_xlabel("Năm")
ax.set_ylabel("Tỷ trọng xuất khẩu (%)")
ax.set_title("Cơ cấu xuất khẩu của Việt Nam theo nhóm ngành (2019–2023)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1),
          fontsize=8, frameon=False)
ax.set_xticklabels(pivot_df.index, rotation=0)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("structure_industry.png", dpi=150, bbox_inches="tight")
plt.show()

Xét theo nhóm ngành lớn, cơ cấu xuất khẩu của Việt Nam trong giai đoạn 2019–2023 thể hiện sự phân hóa rõ rệt và tương đối ổn định giữa các nhóm ngành qua từng năm. Nhóm Máy móc – Điện tử giữ vị trí chi phối tuyệt đối trong suốt cả giai đoạn, với tỷ trọng dao động từ 41,7% năm 2019 lên đến 46,7% năm 2023. Đây không chỉ là nhóm ngành lớn nhất mà còn là nhóm duy nhất có xu hướng tỷ trọng tăng đều qua các năm, phản ánh sự mở rộng liên tục của khu vực sản xuất điện tử và máy móc tại Việt Nam, chủ yếu được dẫn dắt bởi các tập đoàn FDI lớn. Đến năm 2023, gần một nửa tổng kim ngạch xuất khẩu quốc gia đến từ nhóm ngành này — một mức độ tập trung đáng chú ý xét từ góc độ quản trị rủi ro.

Nhóm Dệt may – Da giày đứng thứ hai nhưng theo chiều ngược lại: tỷ trọng giảm từ 23,8% năm 2019 xuống còn 18,6% năm 2023, mức giảm gần 5 điểm phần trăm trong vòng năm năm. Xu hướng này phản ánh áp lực cạnh tranh ngày càng lớn từ các nước xuất khẩu có chi phí thấp hơn, cùng với sự suy yếu của nhu cầu hàng may mặc và giày dép tại các thị trường lớn như Mỹ và EU trong giai đoạn hậu đại dịch. Các nhóm còn lại duy trì tỷ trọng tương đối ổn định và nhỏ hơn đáng kể. Nông – Thủy sản dao động quanh mức 6,2–7,5%, Gỗ – Giấy – Nội thất khoảng 5,3–6,0%, Hóa chất & Nhựa và Kim loại mỗi nhóm chiếm khoảng 5–5,5%. Các nhóm còn lại như Khoáng sản & Năng lượng, Phương tiện vận tải, Thực phẩm chế biến đều dưới 2,5% và không có biến động đáng kể.

Nhìn tổng thể, cơ cấu xuất khẩu theo nhóm ngành của Việt Nam trong giai đoạn nghiên cứu thể hiện tính ổn định tương đối cao, khi tỷ trọng của các nhóm ngành lớn không có sự biến động đột ngột qua các năm. Tuy nhiên, xu hướng tỷ trọng của nhóm Máy móc – Điện tử tiếp tục gia tăng trong khi nhóm Dệt may – Da giày có xu hướng giảm dần cho thấy cơ cấu xuất khẩu đang từng bước chuyển dịch theo hướng gia tăng tỷ trọng các ngành công nghiệp chế biến có hàm lượng công nghệ cao. Đối với ngân hàng, mức độ tập trung ngày càng lớn của xuất khẩu vào nhóm Máy móc – Điện tử cũng đồng nghĩa với việc các biến động trong chuỗi cung ứng điện tử toàn cầu, chẳng hạn như gián đoạn nguồn cung linh kiện, biến động nhu cầu công nghệ hoặc thay đổi chính sách thương mại quốc tế, có thể ảnh hưởng đáng kể đến hoạt động xuất khẩu của Việt Nam

### 4.2.2 Cơ cấu theo HS2 (Top ngành theo từng năm)  

Sau khi xem xét cơ cấu xuất khẩu ở cấp độ nhóm ngành lớn, bước tiếp theo là phân tích cơ cấu xuất khẩu ở cấp độ chi tiết hơn theo mã HS2. Việc phân tích theo HS2 cho phép xác định cụ thể những ngành hàng nào đóng vai trò chủ lực trong tổng kim ngạch xuất khẩu của Việt Nam, đồng thời giúp quan sát rõ hơn sự thay đổi vị trí của các ngành xuất khẩu lớn qua từng năm.

Trên cơ sở dữ liệu đã được tổng hợp theo năm và theo mã HS2, các ngành có giá trị xuất khẩu lớn nhất trong từng năm được xác định, đồng thời tính toán tỷ trọng xuất khẩu của từng ngành trong tổng kim ngạch xuất khẩu của năm tương ứng. Kết quả này cho phép nhận diện các ngành xuất khẩu dẫn đầu và đánh giá mức độ tập trung của cơ cấu xuất khẩu ở cấp độ ngành hàng.

In [ ]:
# Gắn mô tả HS2
df_hs = dfA.merge(
    hs_list,
    on="HS2",
    how="left"
)

df_hs.head()

In [ ]:
# Tổng xuất khẩu theo năm
total_year = df_hs.groupby("Year")["Total_Export_Value"].transform("sum")

# Tỷ trọng HS2 trong năm
df_hs["Export_Share_%"] = df_hs["Total_Export_Value"] / total_year * 100

top_hs_each_year = (
    df_hs
    .sort_values(["Year", "Total_Export_Value"], ascending=[True, False])
    .groupby("Year")
    .head(5)
    .reset_index(drop=True)
)

top_hs_each_year[
    ["Year", "HS2", "HS_Desc", "Total_Export_Value", "Export_Share_%"]
]

Khi đi sâu xuống cấp độ chi tiết hơn theo mã HS2, cơ cấu xuất khẩu của Việt Nam tiếp tục thể hiện mức độ tập trung cao vào một số ngành hàng cụ thể. Trong toàn bộ giai đoạn 2019–2023, mã HS85 (Electrical machinery and equipment) luôn giữ vị trí ngành xuất khẩu lớn nhất. Tỷ trọng của HS85 dao động từ khoảng 36,7% năm 2019 đến gần 39,5% năm 2020, sau đó duy trì quanh mức 37–39% trong các năm tiếp theo, cho thấy vai trò chi phối của ngành điện tử và thiết bị điện trong cơ cấu xuất khẩu của Việt Nam.

Đứng thứ hai trong hầu hết các năm là mã HS84 (Machinery and mechanical appliances), với tỷ trọng tăng dần từ khoảng 4,9% năm 2019 lên hơn 9% vào năm 2023. Xu hướng này cho thấy sự mở rộng của các ngành sản xuất máy móc và thiết bị cơ khí, phản ánh quá trình phát triển của khu vực công nghiệp chế biến và lắp ráp tại Việt Nam.

Bên cạnh đó, các ngành thuộc nhóm dệt may và da giày như HS64, HS61 và HS62  thường xuyên xuất hiện trong nhóm các ngành xuất khẩu lớn theo từng năm. Tuy nhiên, tỷ trọng của các ngành này có xu hướng giảm nhẹ hoặc biến động trong khoảng 4–7%, cho thấy mặc dù vẫn giữ vai trò quan trọng trong cơ cấu xuất khẩu, nhóm ngành này đang dần nhường vị trí cho các ngành công nghiệp chế biến có hàm lượng công nghệ cao hơn. 

Nhìn chung, kết quả phân tích theo HS2 cho thấy cơ cấu xuất khẩu của Việt Nam có mức độ tập trung cao vào một số ngành công nghiệp chủ lực, đặc biệt là các ngành điện tử và máy móc. Điều này phù hợp với xu hướng chuyển dịch cơ cấu xuất khẩu của Việt Nam trong những năm gần đây, khi các ngành công nghiệp chế biến, chế tạo đóng vai trò ngày càng lớn trong tổng kim ngạch xuất khẩu. Từ góc độ phân tích dữ liệu và quản trị rủi ro, việc xác định các ngành HS2 chiếm tỷ trọng lớn trong xuất khẩu là cơ sở quan trọng để đánh giá mức độ tập trung ngành và mức độ ổn định của dòng xuất khẩu, từ đó phục vụ cho các phân tích tiếp theo về mức độ biến động và rủi ro ngành.

## 4.3. Phân tích mức độ tập trung ngành

Sau khi phân tích cơ cấu xuất khẩu theo nhóm ngành và theo mã HS2, bước tiếp theo là đánh giá mức độ tập trung của hoạt động xuất khẩu vào một số ngành hàng chủ lực. Việc xuất khẩu tập trung quá nhiều vào một số ít ngành có thể làm gia tăng rủi ro cho nền kinh tế cũng như cho các tổ chức tài chính tham gia tài trợ thương mại.

Để đánh giá mức độ tập trung ngành, hai chỉ số phổ biến được sử dụng là **CR (Concentration Ratio)** và **HHI (Herfindahl–Hirschman Index).** Chỉ số CR5 và CR10 phản ánh tỷ trọng tích lũy của 5 và 10 ngành HS2 lớn nhất trong tổng kim ngạch xuất khẩu mỗi năm, qua đó cho thấy mức độ chi phối của một nhóm nhỏ ngành hàng đối với toàn bộ hoạt động xuất khẩu. Trong khi đó, HHI được tính bằng tổng bình phương tỷ trọng của tất cả các ngành và phản ánh mức độ tập trung của toàn bộ cơ cấu ngành.

Theo thông lệ quốc tế, HHI < 0,15 cho thấy cơ cấu phân tán, 0,15–0,25 thể hiện mức độ tập trung trung bình và > 0,25 phản ánh mức độ tập trung cao. Khác với CR chỉ xem xét nhóm ngành lớn nhất, HHI phản ánh toàn bộ phân phối ngành và thường được sử dụng trong đánh giá rủi ro tập trung danh mục trong lĩnh vực tài chính và ngân hàng.

In [ ]:
def concentration_ratio(df_year, top_n=10):
    df_sorted = df_year.sort_values("Total_Export_Value", ascending=False)
    top_sum = df_sorted.head(top_n)["Total_Export_Value"].sum()
    total = df_sorted["Total_Export_Value"].sum()
    return top_sum / total * 100

cr_list = []
for y, g in dfA.groupby("Year"):
    cr10 = concentration_ratio(g, top_n=10)
    cr5 = concentration_ratio(g, top_n=5)
    cr_list.append({"Year": y, "CR5_%": cr5, "CR10_%": cr10})

cr_df = pd.DataFrame(cr_list).sort_values("Year")
cr_df

Kết quả tính toán cho thấy mức độ tập trung ngành của xuất khẩu Việt Nam trong giai đoạn 2019–2023 ở mức khá cao và tương đối ổn định theo thời gian. Chỉ số CR5 – phản ánh tỷ trọng của 5 ngành HS2 lớn nhất – dao động quanh mức 60–62% trong toàn bộ giai đoạn nghiên cứu. Điều này cho thấy chỉ 5 ngành hàng lớn nhất đã chiếm khoảng ba phần năm tổng kim ngạch xuất khẩu, phản ánh sự phụ thuộc đáng kể của hoạt động xuất khẩu vào một số ngành chủ lực.

Khi mở rộng phạm vi sang 10 ngành lớn nhất, chỉ số CR10 dao động trong khoảng 72–74%, nghĩa là 10 ngành HS2 hàng đầu chiếm gần ba phần tư tổng giá trị xuất khẩu. Mức độ tập trung này duy trì khá ổn định qua các năm, cho thấy cấu trúc xuất khẩu của Việt Nam không có sự phân tán mạnh sang các ngành mới trong giai đoạn nghiên cứu.

In [ ]:
# Thêm HHI vào cr_df
hhi_list = []
for y, g in dfA.groupby("Year"):
    total = g["Total_Export_Value"].sum()
    shares = g["Total_Export_Value"] / total
    hhi = (shares ** 2).sum()
    hhi_list.append({"Year": y, "HHI": round(hhi, 4)})

hhi_df = pd.DataFrame(hhi_list)
cr_df = cr_df.merge(hhi_df, on="Year")
print(cr_df)

# Vẽ
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
ax.plot(cr_df["Year"], cr_df["CR5_%"],  marker="o", label="CR5 (%)")
ax.plot(cr_df["Year"], cr_df["CR10_%"], marker="s", label="CR10 (%)")
ax2 = ax.twinx()
ax2.plot(cr_df["Year"], cr_df["HHI"], marker="^",
         color="gray", linestyle="--", label="HHI (trục phải)")
ax.set_xlabel("Năm")
ax.set_ylabel("Tỷ lệ tập trung (%)")
ax2.set_ylabel("HHI")
ax.set_title("Mức độ tập trung ngành xuất khẩu Việt Nam (2019–2023)")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("concentration_cr_hhi.png", dpi=150, bbox_inches="tight")
plt.show()

Chỉ số HHI cung cấp thêm một góc nhìn thú vị hơn. Năm 2019, HHI ở mức 0,1544 - nằm trong ngưỡng tập trung vừa theo thang phân loại quốc tế. Chỉ số này tăng lên 0,1739 vào năm 2020, tức là mức độ tập trung thực sự tăng lên trong năm đại dịch, phản ánh việc các ngành nhỏ bị ảnh hưởng nặng nề hơn trong khi nhóm Máy móc - Điện tử vẫn duy trì được quy mô xuất khẩu. Từ năm 2021 trở đi, HHI có xu hướng giảm dần - từ 0,1713 xuống 0,1634 năm 2022 và 0,1620 năm 2023 - cho thấy khi kinh tế phục hồi, các ngành nhỏ hơn dần lấy lại tỷ trọng và cơ cấu xuất khẩu trở nên phân tán hơn một chút. Tuy nhiên, mức HHI hiện tại vẫn nằm trong vùng tập trung vừa và chưa có dấu hiệu giảm về vùng phân tán.

Sự khác biệt giữa xu hướng của CR và HHI cũng đáng chú ý: trong khi CR5 và CR10 gần như phẳng qua các năm, HHI lại có diễn biến rõ hơn - tăng đỉnh năm 2020 rồi giảm dần. Điều này cho thấy phần thay đổi không nằm ở nhóm top mà nằm ở phần đuôi của phân phối: các ngành nhỏ và vừa bị thu hẹp trong đại dịch rồi mở rộng trở lại sau đó, trong khi top 5-10 ngành dẫn đầu gần như không đổi. Từ góc độ quản trị rủi ro ngân hàng, mức HHI duy trì trên 0,15 trong suốt giai đoạn nghiên cứu hàm ý rằng rủi ro tập trung danh mục LC theo ngành là có thực và cần được kiểm soát chủ động. Theo khuyến nghị của Ủy ban Basel (BCBS, 2006), rủi ro tập trung theo ngành cần được phản ánh vào chính sách hạn mức ngành và được theo dõi định kỳ - đặc biệt trong bối cảnh nhóm Máy móc - Điện tử tiếp tục gia tăng tỷ trọng như đã phân tích ở mục 4.2.

## 4.4. Phân tích mức độ ổn định và biến động theo ngành

Trong khi các phần trước tập trung phân tích quy mô và cơ cấu xuất khẩu, phần này đi sâu vào khía cạnh thứ ba của hoạt động xuất khẩu: mức độ ổn định của từng ngành theo thời gian. Một ngành có quy mô lớn chưa chắc đã có mức độ ổn định cao, và ngược lại, một số ngành quy mô nhỏ có thể có tốc độ tăng trưởng rất biến động giữa các năm.

Từ góc độ phân tích rủi ro, các ngành có mức biến động cao thường tiềm ẩn rủi ro lớn hơn trong hoạt động tài trợ thương mại, bởi sự thay đổi mạnh của giá trị xuất khẩu có thể phản ánh sự phụ thuộc vào nhu cầu thị trường, biến động giá quốc tế hoặc các yếu tố chu kỳ kinh tế. Do đó, việc đo lường mức độ biến động tăng trưởng của từng ngành là bước quan trọng để đánh giá mức độ ổn định của cơ cấu xuất khẩu.

### 4.4.1. Tính tốc độ tăng trưởng YoY theo HS2

Để đánh giá mức độ biến động của các ngành xuất khẩu, bước đầu tiên là tính tốc độ tăng trưởng xuất khẩu theo năm (Year-over-Year Growth – YoY) cho từng mã HS2. Chỉ số YoY được xác định bằng tỷ lệ thay đổi của giá trị xuất khẩu giữa hai năm liên tiếp, qua đó phản ánh mức độ tăng hoặc giảm của xuất khẩu theo thời gian. Do tốc độ tăng trưởng YoY được tính dựa trên sự thay đổi giữa các năm liên tiếp, nên năm đầu tiên trong giai đoạn nghiên cứu không có giá trị YoY.

In [ ]:
df_yoy = dfA.sort_values(["HS2", "Year"]).copy()

df_yoy["YoY_Growth_%"] = (
    df_yoy.groupby("HS2")["Total_Export_Value"]
          .pct_change() * 100
)

df_yoy.head(10)


Kết quả tính toán cho thấy tốc độ tăng trưởng xuất khẩu theo năm ở cấp độ HS2 có mức độ biến động khá lớn giữa các ngành và giữa các năm. Một số ngành ghi nhận sự thay đổi mạnh về tốc độ tăng trưởng, bao gồm cả các năm tăng trưởng rất cao và các năm suy giảm đáng kể. Ví dụ, đối với mã HS 01 (Live Animals), tốc độ tăng trưởng YoY giảm mạnh −42,39% vào năm 2020, sau đó tăng trở lại 91,90% vào năm 2021 và tiếp tục tăng mạnh trong các năm tiếp theo. Mức biến động lớn như vậy cho thấy giá trị xuất khẩu của ngành này có sự phụ thuộc đáng kể vào các yếu tố thị trường hoặc điều kiện thương mại trong từng năm.

Tương tự, một số ngành khác như HS 02 (Meat and edible meat offal) cũng ghi nhận mức tăng trưởng âm trong giai đoạn 2020, sau đó phục hồi với mức tăng trưởng dương trong các năm tiếp theo. Những biến động này phản ánh tác động của các yếu tố bên ngoài như sự gián đoạn chuỗi cung ứng, thay đổi nhu cầu thị trường và các biến động thương mại trong giai đoạn chịu ảnh hưởng của đại dịch COVID-19. Nhìn chung, kết quả phân tích YoY cho thấy tốc độ tăng trưởng xuất khẩu giữa các ngành HS2 không đồng đều. Một số ngành có xu hướng tăng trưởng ổn định qua các năm, trong khi các ngành khác có mức biến động lớn. Chỉ số YoY do đó đóng vai trò là cơ sở quan trọng để tiếp tục đo lường mức độ biến động tăng trưởng của từng ngành, được thực hiện trong bước phân tích tiếp theo.

### 4.4.2 Đo độ biến động YoY của từng ngành

Sau khi tính tốc độ tăng trưởng xuất khẩu theo năm (YoY) cho từng mã HS2, bước tiếp theo là đo lường mức độ biến động của tốc độ tăng trưởng này theo thời gian. Trong nghiên cứu này, mức độ biến động được đo bằng độ lệch chuẩn (standard deviation) của chỉ số YoY Growth của từng ngành trong giai đoạn 2019–2023. 

Chỉ số này, được ký hiệu là YoY Volatility, phản ánh mức độ ổn định của tăng trưởng xuất khẩu: ngành có độ lệch chuẩn càng lớn thì tốc độ tăng trưởng giữa các năm càng biến động mạnh. Ngược lại, ngành có giá trị volatility thấp thường có tốc độ tăng trưởng ổn định hơn theo thời gian.

In [ ]:
volatility_by_hs = (
    df_yoy.groupby("HS2", as_index=False)
          .agg(
              Avg_Export=("Total_Export_Value", "mean"),
              YoY_Volatility=("YoY_Growth_%", "std")
          )
          .sort_values("YoY_Volatility", ascending=False)
)

volatility_by_hs.head(15)


Kết quả đo độ biến động YoY theo từng mã HS2 cho thấy sự phân hóa rất lớn giữa các ngành, với khoảng cách giữa ngành biến động nhất và ngành ổn định nhất trải dài hàng nghìn điểm phần trăm. Đứng đầu danh sách là HS 93 với YoY_Volatility lên tới 6.207 - một con số cực đoan nhưng không đáng ngạc nhiên, vì đây là nhóm hàng đặc thù, khối lượng giao dịch rất nhỏ (Avg_Export chỉ khoảng 149.000 USD) và phụ thuộc hoàn toàn vào các hợp đồng nhà nước không thường xuyên. Tương tự, HS 43 và HS 91 cũng có biến động rất cao với volatility lần lượt là 955 và 302, đều là những ngành có quy mô nhỏ và thị trường ngách hẹp.

Điểm đáng chú ý hơn là HS 47 với volatility 183 và Avg_Export khoảng 40 triệu USD, đây là mức biến động cao với quy mô không hề nhỏ, phản ánh sự phụ thuộc mạnh vào chu kỳ giá bột giấy toàn cầu vốn rất không ổn định. HS 01 với volatility 99 cũng phù hợp với nhận xét đã nêu ở mục 4.4.1 - nhóm nông sản sống chịu tác động lớn từ chính sách biên mậu và kiểm dịch.
Một đặc điểm nổi bật xuyên suốt bảng này là: tất cả 15 ngành có biến động cao nhất đều có Avg_Export ở mức nhỏ đến trung bình, không có ngành nào trong top 15 có quy mô xuất khẩu lớn. Điều này ngầm xác nhận nhận định từ phân tích cơ cấu: các ngành chủ lực như HS 85, HS 84 hay nhóm dệt may tuy chiếm tỷ trọng lớn nhưng lại có tăng trưởng tương đối ổn định hơn nhiều nhờ vào chuỗi cung ứng trưởng thành, hợp đồng dài hạn với đối tác quốc tế và lượng khách hàng đa dạng.

Đối với nghiệp vụ LC, các ngành có YoY_Volatility cao là tín hiệu cảnh báo trực tiếp: doanh nghiệp xuất khẩu trong nhóm này có thể có đơn hàng năm này nhưng không có năm sau, khiến khả năng thực hiện hợp đồng và xuất trình chứng từ LC đúng hạn trở nên khó dự đoán hơn. Ngân hàng nên cân nhắc yêu cầu ký quỹ cao hơn hoặc thẩm định kỹ hơn về lịch sử giao dịch khi xử lý LC cho các doanh nghiệp thuộc các ngành này. Do chuỗi thời gian gồm 5 năm (2019–2023), mỗi ngành chỉ có tối đa 4 giá trị YoY để tính độ lệch chuẩn — cỡ mẫu nhỏ khiến các chỉ số biến động có sai số ước lượng tương đối lớn. Giai đoạn 2019–2023 được chọn có chủ đích vì bao phủ đủ ba pha kinh tế rõ ràng (trước, trong và sau COVID-19), giúp đánh giá được khả năng chống chịu và phục hồi của từng ngành — điều mà giai đoạn bình thường thuần túy không thể cho thấy. Các chỉ số biến động trong nghiên cứu này do đó mang ý nghĩa so sánh tương đối giữa các ngành trong cùng giai đoạn, và sẽ được cập nhật khi có thêm dữ liệu.

### 4.4.3. Phân loại nhóm ngành theo quy mô và mức độ ổn định xuất khẩu

Sau khi tính toán quy mô xuất khẩu trung bình (Avg_Export) và mức độ biến động tăng trưởng (YoY Volatility) của từng nhóm ngành, bước tiếp theo là phân loại các ngành theo hai tiêu chí này nhằm làm rõ đặc điểm cấu trúc của hoạt động xuất khẩu. Cụ thể, trục hoành của biểu đồ thể hiện giá trị xuất khẩu trung bình của từng ngành, trong khi trục tung thể hiện độ biến động của tốc độ tăng trưởng xuất khẩu (YoY Std). Hai giá trị trung vị của các biến này được sử dụng để chia biểu đồ thành bốn vùng (quadrant), từ đó phân loại các ngành theo quy mô và mức độ ổn định của tăng trưởng.

- **Q1 — Chiến lược (lớn & ổn định):** Quy mô lớn, biến động thấp. Đây là nhóm ngành ưu tiên tài trợ LC vì dòng xuất khẩu đều đặn và dự đoán được.
- **Q2 — Quan sát (lớn & biến động):** Quy mô lớn nhưng tăng trưởng dao động mạnh. Ngân hàng nên tài trợ có điều kiện và theo dõi sát diễn biến thị trường.
- **Q3 — Ổn định nhỏ (nhỏ & ổn định):** Quy mô khiêm tốn nhưng tăng trưởng đều. Ít rủi ro nhưng cũng ít ảnh hưởng đến danh mục tổng thể.
- **Q4 — Rủi ro (nhỏ & biến động):** Quy mô nhỏ kết hợp biến động lớn tạo rủi ro kép: nhu cầu thị trường không ổn định và năng lực giao hàng khó dự đoán.

In [ ]:
### 4.4.3. Phân loại ngành theo quadrant (Industry_Group)

df_ig = (
    dfA.groupby(["Year", "Industry_Group"], as_index=False)
       .agg(Group_Export=("Total_Export_Value", "sum"))
)
df_ig_yoy = df_ig.sort_values(["Industry_Group", "Year"]).copy()
df_ig_yoy["YoY_pct"] = (
    df_ig_yoy.groupby("Industry_Group")["Group_Export"]
             .pct_change() * 100
)

# Tổng hợp: avg export + volatility theo Industry_Group
quadrant_df = (
    df_ig_yoy.groupby("Industry_Group", as_index=False)
             .agg(
                 Avg_Export_USD=("Group_Export", "mean"),
                 YoY_Std=("YoY_pct", "std")
             )
)

med_export = quadrant_df["Avg_Export_USD"].median()
med_vol    = quadrant_df["YoY_Std"].median()

def assign_quadrant(row):
    high_export = row["Avg_Export_USD"] >= med_export
    high_vol    = row["YoY_Std"]        >= med_vol
    if high_export and not high_vol:
        return "Q1: Chiến lược (lớn – ổn định)"
    elif high_export and high_vol:
        return "Q2: Quan sát (lớn – biến động)"
    elif not high_export and not high_vol:
        return "Q3: Ổn định nhỏ (nhỏ – ổn định)"
    else:
        return "Q4: Rủi ro (nhỏ – biến động)"

quadrant_df["Quadrant"] = quadrant_df.apply(assign_quadrant, axis=1)

Biểu đồ scatter plot dưới đây trực quan hóa vị trí của từng nhóm ngành trên hai trục phân tích, với đường phân vị làm ranh giới giữa bốn góc phần tư:

In [ ]:
# Vẽ scatter plot
import matplotlib.pyplot as plt

color_map = {
    "Q1: Chiến lược (lớn – ổn định)":   "#1d6fa4",
    "Q2: Quan sát (lớn – biến động)":    "#e08a1e",
    "Q3: Ổn định nhỏ (nhỏ – ổn định)":  "#2e9e6b",
    "Q4: Rủi ro (nhỏ – biến động)":      "#c0392b",
}

fig, ax = plt.subplots(figsize=(11, 7), dpi=150)

for _, row in quadrant_df.iterrows():
    color = color_map[row["Quadrant"]]
    ax.scatter(row["Avg_Export_USD"] / 1e9, row["YoY_Std"],
               color=color, s=140, zorder=3,
               edgecolors="white", linewidths=0.6)
    ax.annotate(
        row["Industry_Group"],
        xy=(row["Avg_Export_USD"] / 1e9, row["YoY_Std"]),
        xytext=(6, 4), textcoords="offset points",
        fontsize=8, color=color
    )

ax.axvline(med_export / 1e9, linestyle="--", color="gray",
           linewidth=0.8, alpha=0.7)
ax.axhline(med_vol, linestyle="--", color="gray",
           linewidth=0.8, alpha=0.7)

# Nhãn 4 góc
ax.text(0.02, 0.97, "Q3: Ổn định nhỏ", transform=ax.transAxes,
        fontsize=8, color="#2e9e6b", va="top")
ax.text(0.98, 0.97, "Q4: Rủi ro cao",  transform=ax.transAxes,
        fontsize=8, color="#c0392b", va="top", ha="right")
ax.text(0.02, 0.03, "Q1: Chiến lược",  transform=ax.transAxes,
        fontsize=8, color="#1d6fa4")
ax.text(0.98, 0.03, "Q2: Quan sát",    transform=ax.transAxes,
        fontsize=8, color="#e08a1e", ha="right")

ax.set_xlabel("Giá trị xuất khẩu trung bình (tỷ USD)")
ax.set_ylabel("Độ biến động tăng trưởng YoY (std %)")
ax.set_title("Phân loại nhóm ngành xuất khẩu theo quy mô và mức độ ổn định\n(Việt Nam, 2019–2023)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("quadrant_industry_group.png", dpi=150, bbox_inches="tight")
plt.show()

# Bảng tóm tắt
quadrant_df[["Industry_Group","Quadrant","Avg_Export_USD","YoY_Std"]]\
    .sort_values("Quadrant")\
    .reset_index(drop=True)

Biểu đồ quadrant và bảng phân loại cho thấy bức tranh phân hóa rõ nét giữa 13 nhóm ngành xuất khẩu của Việt Nam khi đặt đồng thời hai chiều quy mô và ổn định lên cùng một mặt phẳng.

* Nhóm 1: Quy mô lớn – ổn định (Q1: Chiến lược). Nhóm này bao gồm các ngành có giá trị xuất khẩu lớn và mức độ biến động tăng trưởng thấp, tiêu biểu như Máy móc – Điện tử, Dệt may – Da giày, Nông – Thủy sản, Gỗ – Giấy – Nội thất và Thực phẩm chế biến. Đây là những ngành đóng vai trò chủ lực trong cơ cấu xuất khẩu của Việt Nam và đồng thời duy trì mức tăng trưởng tương đối ổn định qua các năm. Đặc biệt, ngành Máy móc – Điện tử có quy mô xuất khẩu vượt trội so với các ngành còn lại nhưng lại có mức biến động thấp, cho thấy vị thế trung tâm của ngành này trong hoạt động xuất khẩu.

* Nhóm 2: Quy mô lớn – biến động cao (Q2: Quan sát). Nhóm này bao gồm các ngành có quy mô xuất khẩu tương đối lớn nhưng tốc độ tăng trưởng biến động mạnh, điển hình là Hóa chất & Nhựa và Kim loại. Các ngành trong nhóm này có tiềm năng đóng góp lớn cho xuất khẩu nhưng mức độ biến động tăng trưởng cao cho thấy sự nhạy cảm với biến động của thị trường quốc tế, giá nguyên liệu hoặc chu kỳ kinh tế. Do đó, các ngành này cần được theo dõi chặt chẽ khi đánh giá rủi ro.

* Nhóm 3: Quy mô nhỏ – ổn định (Q3: Ổn định nhỏ). Nhóm này bao gồm các ngành có quy mô xuất khẩu nhỏ nhưng tốc độ tăng trưởng tương đối ổn định, ví dụ như Phương tiện vận tải. Mặc dù đóng góp không lớn trong tổng kim ngạch xuất khẩu, các ngành này có mức độ biến động thấp và có thể đóng vai trò bổ sung trong việc đa dạng hóa cơ cấu xuất khẩu.

* Nhóm 4: Quy mô nhỏ – biến động cao (Q4: Rủi ro cao). Nhóm cuối cùng bao gồm các ngành có quy mô xuất khẩu nhỏ và mức độ biến động tăng trưởng cao, như Khoáng sản & Năng lượng, Hàng tiêu dùng, Dụng cụ & Thiết bị chuyên dụng, Đá – Thủy tinh – Vật liệu xây dựng và một số ngành khác. Những ngành này thường có giá trị xuất khẩu không lớn nhưng lại chịu ảnh hưởng mạnh bởi biến động thị trường, dẫn đến sự thay đổi lớn trong tốc độ tăng trưởng giữa các năm.

Nhìn chung, kết quả phân loại cho thấy cơ cấu xuất khẩu của Việt Nam vừa có các ngành chủ lực với quy mô lớn và mức độ ổn định cao, vừa tồn tại một số ngành có mức biến động lớn hơn. Điều này phản ánh đặc điểm đa dạng của cơ cấu xuất khẩu, trong đó các ngành công nghiệp chế biến đóng vai trò trung tâm, trong khi một số ngành quy mô nhỏ hơn có mức độ nhạy cảm cao hơn với biến động thị trường.

## 4.5. Bảng KPI tổng hợp theo nhóm ngành (Industry_Group)

Sau khi phân tích các khía cạnh khác nhau của hoạt động xuất khẩu bao gồm quy mô, cơ cấu ngành, mức độ tập trung và mức độ biến động tăng trưởng, bước tiếp theo là tổng hợp các kết quả này thành một hệ thống chỉ số KPI có thể sử dụng trực tiếp trong đánh giá rủi ro ngành tại ngân hàng. Thay vì xem xét riêng lẻ từng chiều (quy mô, tập trung, biến động), bộ KPI này tích hợp các chiều đó thành các chỉ số đơn nhất, dễ so sánh và hỗ trợ ra quyết định.

| KPI | Tên chỉ số | Ý nghĩa | Thang đo |
|-----|-----------|---------|---------|
| KPI 4 | Avg Export Share (%) | Tầm quan trọng hệ thống của ngành trong 5 năm | % (cao = quan trọng hơn) |
| KPI 5 | ESI - Export Stability Index | Mức độ ổn định dòng xuất khẩu | [0, 1] (gần 1 = ổn định) |
| KPI 6 | Recovery Rate | Khả năng phục hồi sau COVID-19 | Tỷ lệ 2022/2019 |
| KPI 7 | Risk Score | Điểm rủi ro tổng hợp | [0, 1] (cao = rủi ro cao) |

### 4.5.1. Tính toán các chỉ số KPI thành phần

**KPI 4 - Avg Export Share:** Chỉ số Avg Export Share phản ánh tỷ trọng xuất khẩu trung bình của mỗi nhóm ngành trong tổng kim ngạch xuất khẩu giai đoạn 2019–2023. 

Chỉ số này cho biết tầm quan trọng hệ thống của từng ngành trong cơ cấu xuất khẩu quốc gia. Ngành có tỷ trọng cao hơn thường đóng vai trò trung tâm trong hoạt động xuất khẩu và do đó có ảnh hưởng lớn hơn đến danh mục tài trợ thương mại của ngân hàng.

In [ ]:
# Tạo df_ig (dữ liệu theo Industry_Group x Year) 
df_ig = (
    dfA.groupby(["Year", "Industry_Group"], as_index=False)
       .agg(Group_Export=("Total_Export_Value", "sum"))
)

total_year_ig = df_ig.groupby("Year")["Group_Export"].transform("sum")
df_ig["Share_pct"] = df_ig["Group_Export"] / total_year_ig * 100


In [ ]:
# KPI 4 — Avg Export Share (tầm quan trọng ngành) 
kpi_share = (
    df_ig.groupby("Industry_Group", as_index=False)
         .agg(
             Avg_Export_USD=("Group_Export", "mean"),
             Avg_Share_pct=("Share_pct", "mean")
         )
)

**KPI 5 - Export Stability Index (ESI):** Đo mức độ ổn định của tốc độ tăng trưởng xuất khẩu theo công thức: `ESI = 1 / (1 + CV)` với `CV = Std(YoY) / |Mean(YoY)|`.

ESI luôn nằm trong khoảng [0, 1] - giá trị càng gần 1 thể hiện tăng trưởng càng đều và dự đoán được. Công thức này được chọn thay vì `ESI = 1 − CV` vì khi Mean(YoY) gần 0, CV có thể tiến tới vô cực, làm ESI âm mạnh và mất ý nghĩa diễn giải. Công thức `1/(1+CV)` luôn cho kết quả trong [0, 1] bất kể giá trị CV, đảm bảo tính ổn định và dễ so sánh giữa các ngành. Tuy nhiên, cần lưu ý rằng với chỉ 4 điểm dữ liệu YoY, giá trị ESI trong nghiên cứu này nên được hiểu là chỉ số so sánh tương đối giữa các ngành trong cùng giai đoạn, không phải là đánh giá tuyệt đối về mức độ ổn định dài hạn.

In [ ]:
# KPI 5 — ESI dùng công thức 1/(1+CV)
df_ig_yoy = df_ig.sort_values(["Industry_Group", "Year"]).copy()
df_ig_yoy["YoY_pct"] = (
    df_ig_yoy.groupby("Industry_Group")["Group_Export"]
             .pct_change() * 100
)

kpi_esi = (
    df_ig_yoy.groupby("Industry_Group")["YoY_pct"]
             .agg(["mean", "std"])
             .rename(columns={"mean": "YoY_Mean", "std": "YoY_Std"})
             .reset_index()
)
kpi_esi["CV"]  = kpi_esi["YoY_Std"] / kpi_esi["YoY_Mean"].abs().replace(0, np.nan)
kpi_esi["ESI"] = (1 / (1 + kpi_esi["CV"])).fillna(0).clip(0, 1).round(3)

**KPI 6 - Recovery Rate:** Tỷ lệ xuất khẩu năm 2022 so với năm 2019 (Recovery Rate = Export₂₀₂₂ / Export₂₀₁₉).

Năm 2019 được chọn làm gốc vì đây là năm cuối cùng trước COVID-19, phản ánh mức xuất khẩu bình thường. Năm 2022 được chọn làm năm đánh giá phục hồi vì đây là năm hầu hết các ngành đã thoát khỏi gián đoạn chuỗi cung ứng do đại dịch. Phân loại: > 1,3 = Phục hồi mạnh; 1,0–1,3 = Phục hồi; < 1,0 = Chưa phục hồi.

In [ ]:
# KPI 6 — Recovery Rate (2022 vs 2019)
val_by_year = (
    df_ig.pivot(index="Industry_Group", columns="Year", values="Group_Export")
)
kpi_recovery = pd.DataFrame({
    "Industry_Group": val_by_year.index,
    "Recovery_Rate":  (val_by_year[2022] / val_by_year[2019]).round(3)
}).reset_index(drop=True)

kpi_recovery["Recovery_Label"] = kpi_recovery["Recovery_Rate"].apply(
    lambda x: "Phục hồi mạnh" if x > 1.3
              else ("Phục hồi"      if x >= 1.0
              else  "Chưa phục hồi")
)

In [ ]:
# Bảng KPI tổng hợp
kpi_summary = (
    kpi_share
    .merge(kpi_esi[["Industry_Group", "ESI", "YoY_Mean", "YoY_Std"]], on="Industry_Group")
    .merge(kpi_recovery[["Industry_Group", "Recovery_Rate", "Recovery_Label"]], on="Industry_Group")
)

**KPI 7 - Risk Score tổng hợp:** Kết hợp ba KPI trên thành một điểm rủi ro duy nhất theo công thức có trọng số:

`Risk Score = (1 − ESI) × 0,40 + (1 − Share_norm) × 0,30 + (1 − Recovery_norm) × 0,30`

**Lý do chọn trọng số:**
- **ESI (40%) - trọng số cao nhất:** LC là cam kết thanh toán gắn với việc giao hàng thực tế. Ngành xuất khẩu biến động mạnh đồng nghĩa với rủi ro người xuất khẩu không thực hiện được hợp đồng, dẫn đến tranh chấp chứng từ hoặc không xuất trình được bộ chứng từ hợp lệ đúng hạn. Đây là rủi ro trực tiếp nhất với nghiệp vụ LC.
- **Export Share (30%):** Ngành có tỷ trọng lớn thường có hạ tầng thương mại, chuỗi cung ứng và hệ sinh thái tài trợ thương mại trưởng thành hơn, từ đó rủi ro vận hành thấp hơn. Trọng số thứ hai vì đây là yếu tố cấu trúc có ảnh hưởng gián tiếp đến rủi ro LC.
- **Recovery Rate (30%):** Là chỉ báo về tính bền vững dài hạn nhưng mang tính lịch sử, ít ảnh hưởng trực tiếp đến rủi ro của một giao dịch LC cụ thể trong ngắn hạn.

Risk Score được phân loại: Thấp (< 0,35), Trung bình (0,35–0,60),  Cao (> 0,60).

In [ ]:
# KPI 7 — Risk Score (3 yếu tố: Share, ESI, Recovery) 
# Risk Score = (1 - ESI)*0.40 + (1 - Share_norm)*0.30 + (1 - Recovery_norm)*0.30
# Chuẩn hóa từng yếu tố về [0,1] rồi kết hợp
share_norm    = kpi_summary["Avg_Share_pct"] / kpi_summary["Avg_Share_pct"].max()
esi_norm      = kpi_summary["ESI"]  # đã trong [0,1]
recovery_norm = (kpi_summary["Recovery_Rate"] / kpi_summary["Recovery_Rate"].max()).clip(0, 1)

# Risk cao khi share thấp + ESI thấp + phục hồi kém
# Trọng số: Share 30%, ESI 40%, Recovery 30%
kpi_summary["Risk_Score"] = (
    (1 - esi_norm)      * 0.40 +
    (1 - share_norm)    * 0.30 +
    (1 - recovery_norm) * 0.30
).round(3)

kpi_summary["Risk_Level"] = pd.cut(
    kpi_summary["Risk_Score"],
    bins=[0, 0.35, 0.60, 1.0],
    labels=["Thấp", "Trung bình", "Cao"]
)

kpi_summary.sort_values("Risk_Score")[
    ["Industry_Group","Avg_Export_USD","Avg_Share_pct",
     "ESI","Recovery_Rate","Recovery_Label","Risk_Score","Risk_Level"]
]

Bảng KPI tổng hợp cho thấy sự phân hóa rõ rệt về mức độ rủi ro giữa các nhóm ngành xuất khẩu của Việt Nam khi đánh giá đồng thời ba chiều: tầm quan trọng hệ thống, độ ổn định dòng xuất khẩu và khả năng phục hồi sau cú sốc.

* Nhóm **Risk Level = Thấp** chỉ có một đại diện duy nhất là Máy móc – Điện tử với Risk Score 0,249 — thấp hơn đáng kể so với tất cả các nhóm còn lại. Kết quả này đến từ sự kết hợp của ba yếu tố thuận lợi cùng lúc: tỷ trọng xuất khẩu trung bình cao nhất trong tất cả các ngành (45,3%), ESI ở mức 0,515 phản ánh tăng trưởng tương đối đều, và Recovery Rate đạt 1,541 — tức là xuất khẩu năm 2022 đã vượt mức trước COVID tới 54%. Đây là nhóm ngành mà ngân hàng có thể ưu tiên tài trợ LC với hạn mức linh hoạt và điều kiện thẩm định tiêu chuẩn, vì dòng giao dịch lớn, ổn định và đã được kiểm chứng qua cả giai đoạn khủng hoảng.

* Nhóm **Risk Level = Trung bình** gồm 7 nhóm ngành với Risk Score dao động từ 0,464 đến 0,598. Trong đó Phương tiện vận tải và Thực phẩm chế biến có Risk Score thấp nhất trong nhóm này, lần lượt là 0,464 và 0,499, nhờ Recovery Rate cao (1,506 và 1,423) và ESI khá tốt. Đây là hai ngành có thể được xem xét tài trợ LC tương đối thuận lợi dù không đạt ngưỡng Thấp. Ở phía cuối của nhóm Trung bình, Nông – Thủy sản và Kim loại có Risk Score gần chạm ngưỡng Cao (0,580 và 0,598), chủ yếu do ESI thấp — Nông – Thủy sản với ESI 0,477 và Kim loại chỉ 0,307, cho thấy tăng trưởng xuất khẩu của hai ngành này khá bất ổn qua các năm. Ngân hàng nên theo dõi sát diễn biến đơn hàng và giá cả thị trường quốc tế khi xử lý LC cho doanh nghiệp trong hai nhóm này.

* Nhóm **Risk Level = Cao** gồm 5 nhóm với Risk Score từ 0,633 đến 0,839 — đây là nhóm đòi hỏi sự thận trọng cao nhất trong thẩm định LC. Đáng chú ý nhất là Dệt may – Da giày với Risk Score 0,633, bất chấp quy mô xuất khẩu lớn thứ hai toàn bảng (130 tỷ USD). Nguyên nhân chủ yếu đến từ ESI cực thấp ở mức 0,109 — thấp nhất trong toàn bộ 13 nhóm ngành — phản ánh tốc độ tăng trưởng xuất khẩu của ngành này biến động mạnh qua từng năm, và Recovery Rate chỉ đạt 1,188, tức là phục hồi sau COVID chậm hơn nhiều so với nhóm Máy móc – Điện tử. Kết quả này là lời nhắc nhở quan trọng rằng quy mô lớn không đồng nghĩa với rủi ro thấp — độ ổn định của dòng xuất khẩu mới là yếu tố quyết định trong bối cảnh nghiệp vụ LC.

* Ở cuối bảng, Dụng cụ & Thiết bị chuyên dụng có **Risk Score cao nhất** toàn bảng ở mức 0,839, với Recovery Rate chỉ 0,776 — nhóm duy nhất chưa phục hồi về mức xuất khẩu trước COVID sau ba năm. Kết hợp với tỷ trọng nhỏ (1,4%) và ESI thấp (0,071), đây là nhóm ngành ngân hàng cần áp dụng biện pháp kiểm soát chặt chẽ nhất khi xử lý LC: yêu cầu ký quỹ cao hơn mức thông thường, rút ngắn thời hạn hiệu lực LC, và xem xét kỹ lịch sử giao dịch cũng như năng lực tài chính của từng doanh nghiệp xuất khẩu.

Nhìn chung, bảng KPI tổng hợp cho thấy cơ cấu xuất khẩu của Việt Nam vừa có những ngành chủ lực với mức độ ổn định cao, vừa tồn tại các ngành có rủi ro lớn hơn do quy mô nhỏ và biến động mạnh. Việc tổng hợp các chỉ số thành một bảng KPI thống nhất giúp cung cấp một công cụ phân tích đơn giản nhưng hiệu quả để đánh giá rủi ro ngành trong hoạt động tài trợ thương mại. Kết quả phân tích ở chương này là cơ sở để đề xuất các hàm ý quản trị rủi ro ngành trong hoạt động thư tín dụng tại ngân hàng, được trình bày trong chương tiếp theo.

# V. Xuất data - Import Power BI

In [ ]:
# VERIFY 
import pandas as pd

checks = {
    "df_yoy":       ["Year","HS2","Total_Export_Value","YoY_Growth_%"],
    "total_by_year":["Year","Total_Export","YoY_Growth_%"],
    "group_by_year":["Year","Industry_Group","Group_Export","Export_Share_%"],
    "cr_df":        ["Year","CR5_%","CR10_%","HHI"],
    "quadrant_df":  ["Industry_Group","Avg_Export_USD","YoY_Std","Quadrant"],
    "kpi_summary":  ["Industry_Group","ESI","Risk_Score","Risk_Level"],
    "hs_list":      ["HS2","HS_Desc"],
    "dfA":          ["Year","HS2","Total_Export_Value","Industry_Group"],
}

print("KIỂM TRA BIẾN GỐC:")
print("=" * 60)
all_ok = True
for varname, expected_cols in checks.items():
    try:
        df_check = eval(varname)
        missing = [c for c in expected_cols if c not in df_check.columns]
        if missing:
            print(f"⚠️  {varname:20s} {str(df_check.shape):12s} THIẾU CỘT: {missing}")
            all_ok = False
        else:
            print(f" {varname:20s} {str(df_check.shape):12s} OK")
    except NameError:
        print(f"❌ {varname:20s} KHÔNG TỒN TẠI — cần chạy lại notebook")
        all_ok = False

print("=" * 60)
if all_ok:
    print(" TẤT CẢ OK — có thể chạy cell export an toàn")
else:
    print("CÓ VẤN ĐỀ — cần fix trước khi export")

In [ ]:
# ============================================================
# EXPORT DATA FOR POWER BI — FINAL VERSION
# 4 FACT TABLES + 5 DIMENSION TABLES + 1 HELPER TABLE
# - fact_group_by_year đã được gộp vào fact_export
# - dim_industry được giữ lại theo đúng Star Schema đã vẽ
# ============================================================

import os
import re
import pandas as pd

# ------------------------------------------------------------
# 0. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required_objects = {
    "df_yoy": ["Year", "HS2", "Total_Export_Value", "YoY_Growth_%"],
    "dfA": ["Year", "HS2", "Total_Export_Value", "Industry_Group"],
    "hs_list": ["HS2", "HS_Desc"],
    "total_by_year": ["Year", "Total_Export", "YoY_Growth_%"],
    "cr_df": ["Year", "CR5_%", "CR10_%", "HHI"],
    "quadrant_df": ["Industry_Group", "Avg_Export_USD", "YoY_Std", "Quadrant"],
    "kpi_summary": [
        "Industry_Group", "Avg_Export_USD", "Avg_Share_pct",
        "ESI", "YoY_Mean", "YoY_Std", "Recovery_Rate",
        "Recovery_Label", "Risk_Score", "Risk_Level"
    ],
}

print("=" * 70)
print("KIỂM TRA BIẾN ĐẦU VÀO")
print("=" * 70)

all_ok = True

for var_name, cols in required_objects.items():
    if var_name not in globals():
        print(f"❌ {var_name}: chưa tồn tại. Cần chạy lại các cell phía trên.")
        all_ok = False
        continue

    df_check = globals()[var_name]
    missing_cols = [c for c in cols if c not in df_check.columns]

    if missing_cols:
        print(f"⚠️  {var_name}: thiếu cột {missing_cols}")
        all_ok = False
    else:
        print(f"✅ {var_name:22s} {str(df_check.shape):14s} OK")

if not all_ok:
    raise ValueError("Một số biến/cột đầu vào chưa đủ. Vui lòng kiểm tra lại các cell phía trên.")

print("=" * 70)
print("TẤT CẢ BIẾN ĐẦU VÀO ĐÃ SẴN SÀNG")
print("=" * 70)


# ------------------------------------------------------------
# 1. DIM YEAR
# ------------------------------------------------------------

dim_year = pd.DataFrame({
    "Year": [2019, 2020, 2021, 2022, 2023],
    "Year_Label": ["2019", "2020", "2021", "2022", "2023"],
    "Period": ["Pre-COVID", "COVID", "Recovery", "Peak", "Adjustment"],
    "Year_Order": [1, 2, 3, 4, 5],
})


# ------------------------------------------------------------
# 2. DIM INDUSTRY
# Bảng chiều ở cấp HS2: HS2, mô tả hàng hóa, nhóm ngành
# Nối với fact_export qua khóa HS2
# ------------------------------------------------------------

dim_industry = (
    hs_list[["HS2", "HS_Desc"]]
    .drop_duplicates()
    .copy()
)

dim_industry["HS2"] = dim_industry["HS2"].astype(str).str.zfill(2)

hs_to_group = (
    dfA[["HS2", "Industry_Group"]]
    .drop_duplicates()
    .copy()
)

hs_to_group["HS2"] = hs_to_group["HS2"].astype(str).str.zfill(2)

dim_industry = (
    dim_industry
    .merge(hs_to_group, on="HS2", how="left")
    .drop_duplicates()
    .sort_values("HS2")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. DIM INDUSTRY GROUP
# Bảng chiều ở cấp nhóm ngành, dùng cho các fact tổng hợp
# Nối với fact_volatility_quadrant và fact_risk_summary
# ------------------------------------------------------------

dim_industry_group = (
    dim_industry[["Industry_Group"]]
    .drop_duplicates()
    .dropna()
    .sort_values("Industry_Group")
    .reset_index(drop=True)
)

dim_industry_group["Industry_Group_ID"] = range(1, len(dim_industry_group) + 1)


# ------------------------------------------------------------
# 4. DIM QUADRANT
# Bảng chiều phân loại quy mô - biến động
# ------------------------------------------------------------

def extract_quadrant_order(q):
    match = re.search(r"Q(\d)", str(q))
    return int(match.group(1)) if match else None

def build_quadrant_desc(q):
    q_text = str(q)

    if "Q1" in q_text:
        return "Quy mô lớn - biến động thấp"
    if "Q2" in q_text:
        return "Quy mô lớn - biến động cao"
    if "Q3" in q_text:
        return "Quy mô nhỏ - biến động thấp"
    if "Q4" in q_text:
        return "Quy mô nhỏ - biến động cao"

    return "Nhóm phân loại quy mô - biến động"

dim_quadrant = (
    quadrant_df[["Quadrant"]]
    .drop_duplicates()
    .dropna()
    .copy()
)

dim_quadrant["Quadrant_Order"] = dim_quadrant["Quadrant"].apply(extract_quadrant_order)
dim_quadrant["Quadrant_Desc"] = dim_quadrant["Quadrant"].apply(build_quadrant_desc)

dim_quadrant = (
    dim_quadrant
    .sort_values("Quadrant_Order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. DIM RISK LEVEL
# Bảng chiều mức rủi ro, giúp Power BI sort đúng thứ tự
# ------------------------------------------------------------

dim_risk_level = pd.DataFrame({
    "Risk_Level": ["Thấp", "Trung bình", "Cao"],
    "Risk_Order": [1, 2, 3],
    "Risk_Desc": [
        "Rủi ro tương đối thấp",
        "Rủi ro cần theo dõi",
        "Rủi ro cần xem xét thận trọng"
    ]
})


# ------------------------------------------------------------
# 6. FACT EXPORT
# Bảng fact chi tiết Year - HS2
# Dùng chung cho Dashboard 1 và Dashboard 2
# ------------------------------------------------------------

fact_export = df_yoy[[
    "Year", "HS2", "Total_Export_Value", "YoY_Growth_%"
]].copy()

fact_export["HS2"] = fact_export["HS2"].astype(str).str.zfill(2)

# Bổ sung Industry_Group để có thể tổng hợp trực tiếp theo nhóm ngành
fact_export = fact_export.merge(
    dim_industry[["HS2", "Industry_Group"]],
    on="HS2",
    how="left"
)

# Đổi USD sang triệu USD
fact_export["Total_Export_Value_M"] = (
    fact_export["Total_Export_Value"] / 1_000_000
).round(2)

# Kiểm tra YoY outlier trước khi cap
bi_cap = fact_export[
    (fact_export["YoY_Growth_%"] > 500) |
    (fact_export["YoY_Growth_%"] < -100)
][["Year", "HS2", "Industry_Group", "YoY_Growth_%"]].dropna().sort_values(
    "YoY_Growth_%", ascending=False
)

print("\nKIỂM TRA YOY OUTLIER")
print("=" * 70)
print(f"Số bản ghi có YoY ngoài [-100%, +500%]: {len(bi_cap)}")

if len(bi_cap) > 0:
    print("Chi tiết các bản ghi bị cap:")
    print(bi_cap.to_string(index=False))
else:
    print("Không có bản ghi nào bị cap.")

# Cap YoY outlier để tránh làm lệch trục biểu đồ Power BI
fact_export["YoY_Growth_%"] = (
    fact_export["YoY_Growth_%"].round(2).clip(-100, 500)
)

# Tính Export Share theo tổng kim ngạch từng năm
total_per_year = fact_export.groupby("Year")["Total_Export_Value_M"].transform("sum")

fact_export["Export_Share_%"] = (
    fact_export["Total_Export_Value_M"] / total_per_year * 100
).round(2)

fact_export = fact_export[[
    "Year",
    "HS2",
    "Industry_Group",
    "Total_Export_Value_M",
    "YoY_Growth_%",
    "Export_Share_%"
]]


# ------------------------------------------------------------
# 7. HELPER: TOTAL BY YEAR
# Bảng hỗ trợ hiển thị KPI tổng quan
# Không xem là fact chính trong Star Schema
# ------------------------------------------------------------

total_by_year_export = total_by_year.copy()

total_by_year_export["Total_Export_B"] = (
    total_by_year_export["Total_Export"] / 1_000_000_000
).round(2)

total_by_year_export["YoY_Growth_%"] = (
    total_by_year_export["YoY_Growth_%"].round(2)
)

total_by_year_export = total_by_year_export[[
    "Year", "Total_Export_B", "YoY_Growth_%"
]]


# ------------------------------------------------------------
# 8. FACT CONCENTRATION
# CR5, CR10, HHI theo năm
# ------------------------------------------------------------

fact_concentration = cr_df.copy()

fact_concentration["CR5_%"] = fact_concentration["CR5_%"].round(2)
fact_concentration["CR10_%"] = fact_concentration["CR10_%"].round(2)
fact_concentration["HHI"] = fact_concentration["HHI"].round(4)

fact_concentration = fact_concentration[[
    "Year", "CR5_%", "CR10_%", "HHI"
]]


# ------------------------------------------------------------
# 9. FACT VOLATILITY QUADRANT
# Avg Export, YoY Std, ESI, Quadrant theo nhóm ngành
# ------------------------------------------------------------

fact_volatility_quadrant = quadrant_df.copy()

# Bổ sung ESI từ kpi_summary để phục vụ Dashboard 4
fact_volatility_quadrant = fact_volatility_quadrant.merge(
    kpi_summary[["Industry_Group", "ESI"]],
    on="Industry_Group",
    how="left"
)

fact_volatility_quadrant["Avg_Export_M"] = (
    fact_volatility_quadrant["Avg_Export_USD"] / 1_000_000
).round(2)

fact_volatility_quadrant["YoY_Std"] = fact_volatility_quadrant["YoY_Std"].round(2)
fact_volatility_quadrant["ESI"] = fact_volatility_quadrant["ESI"].round(4)

fact_volatility_quadrant = fact_volatility_quadrant[[
    "Industry_Group", "Avg_Export_M", "YoY_Std", "ESI", "Quadrant"
]]


# ------------------------------------------------------------
# 10. FACT RISK SUMMARY
# Bộ độ đo tổng hợp rủi ro ngành
# Recovery_Rate giữ nguyên theo code hiện tại: 2022 / 2019
# ------------------------------------------------------------

fact_risk_summary = kpi_summary.copy()

fact_risk_summary["Avg_Export_M"] = (
    fact_risk_summary["Avg_Export_USD"] / 1_000_000
).round(2)

fact_risk_summary["Avg_Share_pct"] = fact_risk_summary["Avg_Share_pct"].round(2)
fact_risk_summary["ESI"] = fact_risk_summary["ESI"].round(4)
fact_risk_summary["YoY_Mean"] = fact_risk_summary["YoY_Mean"].round(2)
fact_risk_summary["YoY_Std"] = fact_risk_summary["YoY_Std"].round(2)
fact_risk_summary["Recovery_Rate"] = fact_risk_summary["Recovery_Rate"].round(3)
fact_risk_summary["Risk_Score"] = fact_risk_summary["Risk_Score"].round(3)

fact_risk_summary = fact_risk_summary[[
    "Industry_Group",
    "Avg_Export_M",
    "Avg_Share_pct",
    "ESI",
    "YoY_Mean",
    "YoY_Std",
    "Recovery_Rate",
    "Recovery_Label",
    "Risk_Score",
    "Risk_Level"
]]


# ------------------------------------------------------------
# 11. EXPORT ALL TABLES
# ------------------------------------------------------------

exports = {
    # Dimension tables
    "dim_year.csv": dim_year,
    "dim_industry.csv": dim_industry,
    "dim_industry_group.csv": dim_industry_group,
    "dim_quadrant.csv": dim_quadrant,
    "dim_risk_level.csv": dim_risk_level,

    # Fact tables
    "fact_export.csv": fact_export,
    "fact_concentration.csv": fact_concentration,
    "fact_volatility_quadrant.csv": fact_volatility_quadrant,
    "fact_risk_summary.csv": fact_risk_summary,

    # Helper table
    "total_by_year.csv": total_by_year_export,
}

for filename, df_export in exports.items():
    df_export.to_csv(filename, index=False, encoding="utf-8-sig")


# ------------------------------------------------------------
# 12. FINAL CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("KIỂM TRA FILE XUẤT")
print("=" * 70)

for filename, df_export in exports.items():
    size_kb = os.path.getsize(filename) / 1024
    print(f"{filename:35s} {str(df_export.shape):15s} {size_kb:8.1f} KB")

print("\nKIỂM TRA SỐ LIỆU QUAN TRỌNG")
print("=" * 70)

total_5_years_m = fact_export.groupby("Year")["Total_Export_Value_M"].sum().sum()
export_2023_m = fact_export.loc[
    fact_export["Year"] == 2023, "Total_Export_Value_M"
].sum()

print(f"Total XK 5 năm: {total_5_years_m:,.0f} triệu USD")
print(f"KN 2023: {export_2023_m:,.0f} triệu USD")
print(f"Số mã HS2: {fact_export['HS2'].nunique()}")
print(f"Số nhóm ngành: {dim_industry_group['Industry_Group'].nunique()}")

yoy_2023 = total_by_year_export.loc[
    total_by_year_export["Year"] == 2023, "YoY_Growth_%"
].values[0]

print(f"YoY 2023: {yoy_2023}%")

print("\nKIỂM TRA CẤU TRÚC MÔ HÌNH")
print("=" * 70)
print("Fact tables:")
print("- fact_export.csv")
print("- fact_concentration.csv")
print("- fact_volatility_quadrant.csv")
print("- fact_risk_summary.csv")

print("\nDimension tables:")
print("- dim_year.csv")
print("- dim_industry.csv")
print("- dim_industry_group.csv")
print("- dim_quadrant.csv")
print("- dim_risk_level.csv")

print("\nHelper table:")
print("- total_by_year.csv")

print("=" * 70)
print("XUẤT XONG — SẴN SÀNG IMPORT VÀO POWER BI")
print("=" * 70)